In [71]:
import pandas
import openpyxl
import os


In [ ]:
arquivo_pipefy_clientes_usinas = 'C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Clientes_Usinas.xlsx'
arquivo_pipefy_funil_vendas = 'C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Funil_de_Vendas.xlsx'
arquivo_sunne_funil_vendas = 'C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Sunne.xlsx'
arquivo_relatorio_compensacao = None
arquivo_relatorio_status = 'C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Status.xlsx'
arquivo_destino = None

dataframe_pipefy_clientes_usinas = None
dataframe_pipefy_funil_vendas = None
dataframe_sunne_funil_vendas = None
dataframe_status = None
dataframe_clientes_unidade  = None
dataframe_relatorio_compensacao = None

clientes_com_infos = {}
clientes_sem_infos = {}
clientes_desconhecidos = []
lista_geral = []

datas = {"2026/01": "Jan", "2026/02": "Fev", "2026/03": "Mar", "2026/04": "Abr", "2025/05": "Mai", "2025/06": "Jun", "2025/07": "Jul", "2025/08": "Ago", "2025/09": "Set", "2025/10": "Out", "2025/11": "Nov", "2025/12": "Dez"}



In [73]:
def leitura ():

    global dataframe_pipefy_clientes_usinas
    global dataframe_pipefy_funil_vendas
    global dataframe_sunne_funil_vendas
    global dataframe_status

    dataframe_pipefy_clientes_usinas = pandas.read_excel(arquivo_pipefy_clientes_usinas)
    dataframe_pipefy_funil_vendas = pandas.read_excel(arquivo_pipefy_funil_vendas)
    dataframe_sunne_funil_vendas = pandas.read_excel(arquivo_sunne_funil_vendas)
    dataframe_status = pandas.read_excel(arquivo_relatorio_status)

    return dataframe_pipefy_clientes_usinas, dataframe_pipefy_funil_vendas, dataframe_sunne_funil_vendas, dataframe_status

Composição "Pipefy - Funil de Vendas":

1ª Coluna - Usina
2º Coluna - UG
3ª Coluna - UC
4ª Coluna - Energia Contratada (kWh)

Composição "Sunne - Funil de Vendas":

1ª Coluna - Data de Originação
8º Coluna - Consumo
33ª Coluna - Número de Instalação
34ª Coluna - Tensão

In [74]:
def filtro (unidade):

    global dataframe_clientes_unidade

    dataframe_clientes_unidade = dataframe_pipefy_clientes_usinas[dataframe_pipefy_clientes_usinas['UG'] == unidade]
    dataframe_clientes_unidade = dataframe_clientes_unidade[dataframe_clientes_unidade["UC"].isin(dataframe_status["UC do Cliente"])]
    
    return dataframe_clientes_unidade

In [75]:
def dicionario_base ():
    
    global clientes_com_infos
    global clientes_sem_infos

    clientes_com_infos = {}
    clientes_sem_infos = {}
    
    for index, infos in dataframe_clientes_unidade.iterrows():
        if infos["UC"] not in list(dataframe_pipefy_funil_vendas["Número da instalação CEMIG"]):
            clientes_sem_infos[str(infos["UC"])] = {"Consumo Medio": None, "Contrato": int(infos["Energia Contratada (kWh)"])}
        else:
            cliente_info = dataframe_pipefy_funil_vendas[dataframe_pipefy_funil_vendas["Número da instalação CEMIG"]==infos["UC"]]
            clientes_com_infos[str(infos["UC"])] = {"Contrato": int(infos["Energia Contratada (kWh)"]), 
                                                "Consumo Medio": int(cliente_info["Média de consumo"].values[0]),
                                                "Medicoes": {"Jan": int(cliente_info.iloc[0, 2]),
                                                            "Fev": int(cliente_info.iloc[0, 3]),
                                                            "Mar": int(cliente_info.iloc[0, 4]),
                                                            "Abr": int(cliente_info.iloc[0, 5]),
                                                            "Mai": int(cliente_info.iloc[0, 6]),
                                                            "Jun": int(cliente_info.iloc[0, 7]),
                                                            "Jul": int(cliente_info.iloc[0, 8]),
                                                            "Ago": int(cliente_info.iloc[0, 9]),
                                                            "Set": int(cliente_info.iloc[0, 10]),
                                                            "Out": int(cliente_info.iloc[0, 11]),
                                                            "Nov": int(cliente_info.iloc[0, 12]),
                                                            "Dez": int(cliente_info.iloc[0, 13])}}
    
    return clientes_com_infos, clientes_sem_infos

In [76]:
def visualizar_dicionario_cci ():
    
    for chave in clientes_com_infos.keys():
        print("{} {}".format(chave, clientes_com_infos[chave]))

In [77]:
def visualizar_dicionario_csi ():

    for chave in clientes_sem_infos.keys():
        print("{} {}".format(chave, clientes_sem_infos[chave]))

In [78]:
def coletor_informacoes_cci ():

    lista_infos = {}
    contador = 0

    global dataframe_relatorio_compensacao
    global clientes_com_infos

    dataframe_relatorio_compensacao = pandas.read_excel(arquivo_relatorio_compensacao)

    for chave in clientes_com_infos.keys():
        if int(chave) not in list(dataframe_relatorio_compensacao["Instalacao"]):
            pass
        else:
            dataframe_relatorio_unico = dataframe_relatorio_compensacao[dataframe_relatorio_compensacao["Instalacao"] == int(chave)]
            dataframe_relatorio_unico = dataframe_relatorio_unico.drop(columns=["Periodo_uso", "Modalidade", "Quota", "Posto_horario", "Qtd_geracao", "Qtd_compensacao", "Qtd_saldo_ant", "Qtd_transferencia", "Qtd_recebimento", "Qtd_saldo_atual", "Qtd_saldo_exp", "Qtd_prox_saldo_exp", "Per_prox_saldo_exp"])
            dataframe_relatorio_unico = dataframe_relatorio_unico[dataframe_relatorio_unico["Qtd_consumo"] != 0]
            dataframe_relatorio_unico = dataframe_relatorio_unico.sort_values("Qtd_consumo").drop_duplicates(subset = ['Periodo', 'Instalacao', 'Qtd_consumo'], keep="last").reset_index()
            for abv in datas.keys():
                if abv not in list(dataframe_relatorio_unico["Periodo"]):
                    lista_infos[datas[abv]] = 0
                else:
                    lista_infos[datas[abv]] = int(dataframe_relatorio_unico[dataframe_relatorio_unico["Periodo"] == abv]['Qtd_consumo'].values[0])
                    contador += 1

            clientes_com_infos[chave]["Novas_Medicoes"] = lista_infos
            clientes_com_infos[chave]["Contador"] = contador

            lista_infos = {}
            contador = 0

    return clientes_com_infos

In [79]:
def coletor_informacoes_csi ():

    lista_infos = {}
    contador = 0

    global dataframe_relatorio_compensacao
    global clientes_sem_infos

    dataframe_relatorio_compensacao = pandas.read_excel(arquivo_relatorio_compensacao)

    for chave in clientes_sem_infos.keys():
        if int(chave) not in list(dataframe_relatorio_compensacao["Instalacao"]):
            pass
        else:
            dataframe_relatorio_unico = dataframe_relatorio_compensacao[dataframe_relatorio_compensacao["Instalacao"] == int(chave)]
            dataframe_relatorio_unico = dataframe_relatorio_unico.drop(columns=["Periodo_uso", "Modalidade", "Quota", "Posto_horario", "Qtd_geracao", "Qtd_compensacao", "Qtd_saldo_ant", "Qtd_transferencia", "Qtd_recebimento", "Qtd_saldo_atual", "Qtd_saldo_exp", "Qtd_prox_saldo_exp", "Per_prox_saldo_exp"])
            dataframe_relatorio_unico = dataframe_relatorio_unico[dataframe_relatorio_unico["Qtd_consumo"] != 0]
            dataframe_relatorio_unico = dataframe_relatorio_unico.sort_values("Qtd_consumo").drop_duplicates(subset = ['Periodo', 'Instalacao', 'Qtd_consumo'], keep="last").reset_index()
            for abv in datas.keys():
                if abv not in list(dataframe_relatorio_unico["Periodo"]):
                    lista_infos[datas[abv]] = 0
                else:
                    lista_infos[datas[abv]] = int(dataframe_relatorio_unico[dataframe_relatorio_unico["Periodo"] == abv]['Qtd_consumo'].values[0])
                    contador += 1

            clientes_sem_infos[chave]["Novas_Medicoes"] = lista_infos
            clientes_sem_infos[chave]["Contador"] = contador

            lista_infos = {}
            contador = 0

    return clientes_sem_infos

In [80]:
def analise_contratual_cci ():

    somador = 0
    contador = 0

    global clientes_com_infos

    for uc in clientes_com_infos.keys():
        if clientes_com_infos[uc]["Contador"] < 6:
            pass
        if clientes_com_infos[uc]["Contador"] >= 6:
            for competencia in datas.keys():
                if clientes_com_infos[uc]["Novas_Medicoes"][datas[competencia]] != 0:
                    somador += clientes_com_infos[uc]["Novas_Medicoes"][datas[competencia]]
                    contador +=1
                elif clientes_com_infos[uc]["Medicoes"][datas[competencia]] != 0:
                    somador += clientes_com_infos[uc]["Medicoes"][datas[competencia]]
                    contador +=1
                else:
                    pass
            
            nova_media = somador/contador
            clientes_com_infos[uc]["Novo_Consumo_Medio"] = int(nova_media)

            if (clientes_com_infos[uc]["Consumo Medio"] - clientes_com_infos[uc]["Contrato"]) in [30, 50, 100]:
                novo_contrato = nova_media - (clientes_com_infos[uc]["Consumo Medio"] - clientes_com_infos[uc]["Contrato"])
            else:
                novo_contrato = nova_media
            clientes_com_infos[uc]["Novo_Contrato"] = int(novo_contrato)

            if clientes_com_infos[uc]["Contrato"] <= clientes_com_infos[uc]["Novo_Contrato"] * 0.9:
                clientes_com_infos[uc]["Status"] = "Aumentar"
            elif clientes_com_infos[uc]["Contrato"] >= clientes_com_infos[uc]["Novo_Contrato"] * 1.1:
                clientes_com_infos[uc]["Status"] = "Reduzir"
            else:
                clientes_com_infos[uc]["Status"] = "Permanecer"

        somador = 0
        contador = 0

    return clientes_com_infos

In [81]:
def analise_contratual_csi ():

    somador = 0
    contador = 0
    taxa_min_disp = 0
    
    global clientes_sem_infos
    global clientes_desconhecidos

    for uc in clientes_sem_infos.keys():
        if clientes_sem_infos[uc]["Contador"] < 6:
            pass
        if clientes_sem_infos[uc]["Contador"] >= 6:
            for competencia in datas.keys():
                if clientes_sem_infos[uc]["Novas_Medicoes"][datas[competencia]] != 0:
                    somador += clientes_sem_infos[uc]["Novas_Medicoes"][datas[competencia]]
                    contador +=1
            
            nova_media = somador/contador
            clientes_sem_infos[uc]["Novo_Consumo_Medio"] = int(nova_media)

            if int(uc) not in list(dataframe_sunne_funil_vendas["Instalacao"]):
                clientes_desconhecidos.append(int(uc))
            else:
                dataframe_relatorio_unico = dataframe_sunne_funil_vendas[dataframe_sunne_funil_vendas["Instalacao"]==int(uc)]
                dataframe_relatorio_unico = dataframe_relatorio_unico.sort_values("Data de Originacao").drop_duplicates(subset=["Instalacao"], keep="last").reset_index()
                clientes_sem_infos[uc]["Consumo Medio"] = int(dataframe_relatorio_unico["Consumo"].values[0])
                if dataframe_relatorio_unico["Tensão"].values[0] == "Trifásico":
                    taxa_min_disp = 100
                elif dataframe_relatorio_unico["Tensão"].values[0] == "Bifásico":
                    taxa_min_disp = 50
                else:
                    taxa_min_disp = 30

            novo_contrato = nova_media - taxa_min_disp
            clientes_sem_infos[uc]["Novo_Contrato"] = int(novo_contrato)

            if clientes_sem_infos[uc]["Contrato"] <= clientes_sem_infos[uc]["Novo_Contrato"] * 0.9:
                clientes_sem_infos[uc]["Status"] = "Aumentar"
            elif clientes_sem_infos[uc]["Contrato"] >= clientes_sem_infos[uc]["Novo_Contrato"] * 1.1:
                clientes_sem_infos[uc]["Status"] = "Reduzir"
            else:
                clientes_sem_infos[uc]["Status"] = "Permanecer"

        somador = 0
        contador = 0

    return clientes_sem_infos

In [ ]:
def exportar_excel (unidade):

    df_temp = None
    df_temp_sec = None

    lista_infos = []
    

    lista_infos_sec = []
    lista_geral_sec = []

    for chave in clientes_com_infos.keys():
        if clientes_com_infos[chave]["Contador"] >= 6:
            lista_infos.append(int(chave))
            lista_infos.append(clientes_com_infos[chave]["Contrato"])
            lista_infos.append(clientes_com_infos[chave]["Novo_Contrato"])
            lista_infos.append(clientes_com_infos[chave]["Status"])
            lista_infos.append(clientes_com_infos[chave]["Consumo Medio"])
            lista_infos.append("Pipefy")
            lista_geral.append(lista_infos)
        lista_infos = []
    
    for chave in clientes_sem_infos.keys():
        if clientes_sem_infos[chave]["Contador"] >= 6:
            lista_infos.append(int(chave))
            lista_infos.append(clientes_sem_infos[chave]["Contrato"])
            lista_infos.append(clientes_sem_infos[chave]["Novo_Contrato"])
            lista_infos.append(clientes_sem_infos[chave]["Status"])
            lista_infos.append(clientes_sem_infos[chave]["Consumo Medio"])
            lista_infos.append("Sunne")
            lista_geral.append(lista_infos)
        lista_infos = []

    for chave in clientes_com_infos.keys():
        if clientes_com_infos[chave]["Contador"] < 6:
            lista_infos_sec.append(int(chave))
            lista_infos_sec.append(clientes_com_infos[chave]["Contrato"])
            lista_infos_sec.append(clientes_com_infos[chave]["Consumo Medio"])
            lista_infos_sec.append("Pipefy")
            lista_geral_sec.append(lista_infos_sec)
        lista_infos_sec = []
        
    for chave in clientes_sem_infos.keys():
        if clientes_sem_infos[chave]["Contador"] < 6:
            lista_infos_sec.append(int(chave))
            lista_infos_sec.append(clientes_sem_infos[chave]["Contrato"])
            lista_infos_sec.append(clientes_sem_infos[chave]["Consumo Medio"])
            lista_infos_sec.append("Sunne")
            lista_geral_sec.append(lista_infos_sec)
        lista_infos_sec = []

    dataframe_final = pandas.DataFrame(lista_geral, columns=['UC', 'Contrato Atual', 'Contrato Sugerido', 'Status', 'Consumo Medio', 'Base'])
    dataframe_final_sec = pandas.DataFrame(lista_geral_sec, columns=['UC', 'Contrato Atual', 'Consumo Medio', 'Base'])
 
    wb = openpyxl.load_workbook("C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Bases_UGs/Base.xlsx")
    
    # Seleciona a primeira planilha (ws1) e a segunda (ws2)
    ws1 = wb[wb.sheetnames[0]]
    ws2 = wb[wb.sheetnames[1]]
    
    # Cria um DataFrame temporário com as colunas a serem inseridas na planilha 1:
    # "UC", "Documento" e "Cota Final"
    df_temp = dataframe_final[['UC', 'Contrato Atual', 'Contrato Sugerido', 'Status', 'Consumo Medio', 'Base']].copy()
    df_temp_sec = dataframe_final_sec[['UC', 'Contrato Atual', 'Consumo Medio', 'Base']].copy()

    # Conta o número de linhas (número de UC) para essa UG
    n_linhas = df_temp.shape[0]
    o_linhas = df_temp_sec.shape[0]
    
    # Define a linha de início para a inserção dos dados (conforme template da CEMIG: a partir da linha 16)
    start_row = 11
    ws1.insert_rows(start_row, amount=n_linhas)
    
    # Preenche os dados na planilha 1:
    # Coluna A: UC, Coluna B: Documento, Coluna C: Cota Final
    current_row = start_row
    for idx, row in df_temp.iterrows():
        ws1.cell(row=current_row, column=1, value=row["UC"])
        ws1.cell(row=current_row, column=2, value=row["Contrato Atual"])
        ws1.cell(row=current_row, column=3, value=row["Contrato Sugerido"])
        ws1.cell(row=current_row, column=4, value=row["Status"])
        ws1.cell(row=current_row, column=5, value=row["Consumo Medio"])
        ws1.cell(row=current_row, column=6, value=row["Base"])
        current_row += 1

    start_now_sec = 1
    ws2.insert_rows(start_now_sec, amount=o_linhas)

    current_row_sec = start_now_sec
    for idx, row in df_temp_sec.iterrows():
        ws2.cell(row=current_row_sec, column=1, value=row["UC"])
        ws2.cell(row=current_row_sec, column=2, value=row["Contrato Atual"])
        ws2.cell(row=current_row_sec, column=3, value=row["Consumo Medio"])
        ws2.cell(row=current_row_sec, column=4, value=row["Base"])
        current_row_sec += 1

    file_name_exl_cemig = f"{unidade}.xlsx"
    file_path_exl_cemig = os.path.join("C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/Bases_UGs", file_name_exl_cemig)
    
    # Salva o arquivo Excel
    try:
        wb.save(file_path_exl_cemig)
        print(f"Excel GERADO para a UG {unidade}")
    except Exception as e:
        print(f"Erro ao salvar o arquivo {file_path_exl_cemig}: {e}")
        

In [83]:
def Main (unidade):
    print(unidade)
    leitura()
    filtro(unidade)
    dicionario_base()
    #visualizar_dicionario_cci()
    #visualizar_dicionario_csi()
    coletor_informacoes_cci()
    coletor_informacoes_csi()
    #visualizar_dicionario_cci()
    #visualizar_dicionario_csi()
    analise_contratual_cci()
    analise_contratual_csi()
    visualizar_dicionario_cci()
    visualizar_dicionario_csi()
    exportar_excel(unidade)

In [84]:
caminho = 'C:/Users/lucas.conceicao/Relatorio de Compensacao/Ajuste de Contrato/relatorios'

# Verifica se o caminho existe
if not os.path.exists(caminho):
    raise FileNotFoundError(f"Pasta não encontrada: {caminho}")

# Percorre todos os arquivos na pasta e subpastas
for root, dirs, files in os.walk(caminho):
    for filename in files:
        if filename.lower().endswith('.xlsx'):
            arquivo_relatorio_compensacao = os.path.join(root, filename)
            unidade = int(filename.split('_')[2])
            try:
                Main(unidade)
            except Exception as e:
                print(f"Ignorado {filename}: {e}")
                continue



3014109500
3000029933 {'Contrato': 819, 'Consumo Medio': 1010, 'Medicoes': {'Jan': 972, 'Fev': 1055, 'Mar': 1006, 'Abr': 1184, 'Mai': 1035, 'Jun': 1079, 'Jul': 792, 'Ago': 813, 'Set': 936, 'Out': 1201, 'Nov': 1042, 'Dez': 1009}, 'Novas_Medicoes': {'Jan': 819, 'Fev': 868, 'Mar': 0, 'Abr': 1056, 'Mai': 802, 'Jun': 772, 'Jul': 784, 'Ago': 904, 'Set': 1122, 'Out': 630, 'Nov': 1059, 'Dez': 760}, 'Contador': 11, 'Novo_Consumo_Medio': 881, 'Novo_Contrato': 881, 'Status': 'Permanecer'}
3000113034 {'Contrato': 718, 'Consumo Medio': 767, 'Medicoes': {'Jan': 1219, 'Fev': 544, 'Mar': 346, 'Abr': 271, 'Mai': 319, 'Jun': 499, 'Jul': 1102, 'Ago': 1105, 'Set': 946, 'Out': 950, 'Nov': 1076, 'Dez': 833}, 'Novas_Medicoes': {'Jan': 1042, 'Fev': 1258, 'Mar': 0, 'Abr': 495, 'Mai': 308, 'Jun': 260, 'Jul': 386, 'Ago': 636, 'Set': 967, 'Out': 1080, 'Nov': 1224, 'Dez': 1249}, 'Contador': 11, 'Novo_Consumo_Medio': 770, 'Novo_Contrato': 770, 'Status': 'Permanecer'}
3000144075 {'Contrato': 166, 'Consumo Medio': 21